### 전체 브랜드 제품을 하나의 구조로 통일하기 위해 단일 스키마로 표준화하고자 함. 

### CSV 파일을 합치면서 파일명을 그대로 컬럼으로 넣음. 

In [2]:
import pandas as pd
import glob
import os


CSV_PATH = "/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/*.csv"

files = glob.glob(CSV_PATH)

print("찾은 파일 수:", len(files))
print(files)

dfs = []

for path in files:
    brand = os.path.splitext(os.path.basename(path))[0]
    df = pd.read_csv(path)

    df["brand"] = brand
    df["full_name"] = brand + " " + df["상품명"]

    dfs.append(df)

master = pd.concat(dfs, ignore_index=True)
master.to_csv("/workspaces/STUDY-DATA/first_week/data_csv/amore_master.csv",
              index=False,
              encoding="utf-8-sig")

찾은 파일 수: 30
['/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/메이크온.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/에스트라.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/앞바다즈.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/라보에이치.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/롱테이크.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/에스쁘아.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/아모레성수.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/미쟝센.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/아모레퍼시픽.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/아모레베이직.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/한율.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/해피바스.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/온호프.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/홀리추얼.csv', '/workspaces/STUDY-DATA/first_week/data_csv/amore_csv/퍼즐우드.csv', '/workspac

#### Product ID 생성이유: CRM 생성 중 동일 브랜드 내 유사 제품명, 기획 세트, 리필팩 등 다양한 변수들이 존재함. 때문에 단순한 문자열만으로 제품을 구분하면 데이터 관리가 어려움. 또한 추후에 진행할 Embedding과 RAG 안정성 확보와 데이터 병합에 유리하기에 Product ID를 생성함 

In [ ]:
import pandas as pd
import glob
import re
## 상대경로
input_files = glob.glob("../first_week/data_csv/amore_csv/*.csv")

dfs = []

def clean_name(x):
    if pd.isna(x):
        return x
    x = re.sub(r"\(.*?\)", "", str(x))
    x = re.sub(r"\[.*?\]", "", str(x))
    x = re.sub(r"(?i)sold out|품절", "", str(x))
    return x.strip()

def to_number(x):
    if pd.isna(x):
        return None
    x = re.sub(r"[^0-9.]", "", str(x))
    return float(x) if x else None

def to_rate(x):
    if pd.isna(x):
        return None
    x = re.sub(r"[^0-9.]", "", str(x))
    return float(x) / 100 if x else None
# 가격을 할인율로 가져와 다시 정가 계산을 함. 
def calc_original(price, rate):
    if price is None:
        return None
    if rate is None or rate == 0:
        return price
    try:
        return round(price / (1 - rate))
    except:
        return price

def parse_volume_value(x):
    if pd.isna(x):
        return None
    m = re.search(r"(\d+(\.\d+)?)", str(x))
    return float(m.group(1)) if m else None


def parse_volume_unit(x):
    if pd.isna(x):
        return None
    m = re.search(r"(ml|mL|ML|g|G|kg|KG|l|L|ea|EA|p|P|개|포|정|매|캡슐|병)", str(x))
    return m.group(1) if m else None

def merge_ingredients(row):
    vals = []
    for col in ["전성분", "원재료명및함량", "원재료명 및 함량", "영양성분"]:
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]))
    return " / ".join(vals) if vals else None


for file in input_files:
    df = pd.read_csv(file)


    if "상품명" in df:
        df["상품명"] = df["상품명"].apply(clean_name)

    df["price_current"] = df["가격"].apply(to_number) if "가격" in df else None
    df["discount_rate"] = df["할인율"].apply(to_rate) if "할인율" in df else None

    df["price_original"] = df.apply(
        lambda r: calc_original(r["price_current"], r["discount_rate"]),
        axis=1
    )

    df["용량_raw"] = df["용량"] if "용량" in df else None
    df["용량_value"] = df["용량_raw"].apply(parse_volume_value)
    df["용량_unit"] = df["용량_raw"].apply(parse_volume_unit)

    df["전성분_통합"] = df.apply(merge_ingredients, axis=1)
    df["전성분"] = df["전성분_통합"]

    keep_cols = [
        "brand",
        "product_id",
        "상품명",
        "URL",
        "price_original",
        "용량_raw",
        "용량_value",
        "용량_unit",
        "전성분"
    ]

    existing_cols = [c for c in keep_cols if c in df.columns]

    df = df[existing_cols]

    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

final_df.to_csv("amore_final.csv", index=False)

print("완료: /mnt/data/amore_final.csv 생성됨")

완료: /mnt/data/amore_final.csv 생성됨


/tmp/ipykernel_4716/3617443998.py:127: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(dfs, ignore_index=True)
